# Data Cleaning and Preprocessing

Imports

In [1]:
from pathlib import Path
import pandas as pd

from src.utils.paths import find_project_root, resolve_project_path
from src.utils.config import load_project_configs
from src.data.load_data import load_raw_dataset
from src.data.validate_data import build_missing_summary, build_column_summary, build_dataset_summary

from src.data.clean_data import (
    load_cleaning_context,
    get_schema_details,
    detect_missing_and_invalid_values,
    decide_columns_to_drop,
    mark_invalid_xyz_as_missing,
    remove_impossible_core_rows,
    impute_missing_values,
    clean_diamonds_dataset,
)

from src.data.preprocess import (
    split_numeric_and_categorical,
    summarize_preprocessing_strategy,
    build_regression_preprocessor,
    build_clustering_preprocessor,
)

In [2]:
project_root = find_project_root()
configs = load_project_configs(project_root)

features_config = configs["features_config"]
paths_config = configs["paths_config"]

schema = get_schema_details(features_config)
raw_df = load_raw_dataset(project_root)

print("Project root:", project_root)
print("Raw shape:", raw_df.shape)
display(raw_df.head())

Project root: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation
Raw shape: (53940, 10)


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [3]:
dataset_summary = build_dataset_summary(
    raw_df,
    target_col = schema["target_col"],
    required_columns = schema["required_columns"],
    expected_columns = schema["expected_columns"],
)

dataset_summary

{'shape': {'rows': 53940, 'columns': 10},
 'columns': ['carat',
  'cut',
  'color',
  'clarity',
  'depth',
  'table',
  'price',
  'x',
  'y',
  'z'],
 'target_present': True,
 'missing_required_columns': [],
 'unexpected_columns': [],
 'duplicate_rows': 146,
 'dtypes': {'carat': 'float64',
  'cut': 'object',
  'color': 'object',
  'clarity': 'object',
  'depth': 'float64',
  'table': 'float64',
  'price': 'int64',
  'x': 'float64',
  'y': 'float64',
  'z': 'float64'}}

In [4]:
column_summary = build_column_summary(raw_df)
display(column_summary)

,column,dtype,non null_count,null_count,null_pct,n_unique,sample_values
0,carat,float64,53940,0,0.0,273,"[0.23, 0.21, 0.23]"
1,cut,object,53940,0,0.0,5,"[Ideal, Premium, Good]"
2,color,object,53940,0,0.0,7,"[E, E, E]"
3,clarity,object,53940,0,0.0,8,"[SI2, SI1, VS1]"
4,depth,float64,53940,0,0.0,184,"[61.5, 59.8, 56.9]"
5,table,float64,53940,0,0.0,127,"[55.0, 61.0, 65.0]"
6,price,int64,53940,0,0.0,11602,"[326, 326, 327]"
7,x,float64,53940,0,0.0,554,"[3.95, 3.89, 4.05]"
8,y,float64,53940,0,0.0,552,"[3.98, 3.84, 4.07]"
9,z,float64,53940,0,0.0,375,"[2.43, 2.31, 2.31]"


In [5]:
missing_summary_before = build_missing_summary(raw_df)
display(missing_summary_before)

,column,missing_count,missing_pct
0,carat,0,0.0
1,clarity,0,0.0
2,color,0,0.0
3,cut,0,0.0
4,depth,0,0.0
5,price,0,0.0
6,table,0,0.0
7,x,0,0.0
8,y,0,0.0
9,z,0,0.0


In [6]:
print(type(schema["numeric_cols"]), schema["numeric_cols"])
print(type(schema["categorical_cols"]), schema["categorical_cols"])
print(type(schema["target_col"]), schema["target_col"])
print(type(schema["dimension_columns"]), schema["dimension_columns"])

for c in schema["dimension_columns"]:
    print("DIM COL:", c, type(c), type(raw_df[c]))

for c in schema["numeric_cols"] + [schema["target_col"]]:
    print("NUM CHECK COL:", c, type(c), type(raw_df[c]))

<class 'list'> ['carat', 'depth', 'table', 'x', 'y', 'z']
<class 'list'> ['cut', 'color', 'clarity']
<class 'str'> price
<class 'list'> ['x', 'y', 'z']
DIM COL: x <class 'str'> <class 'pandas.core.series.Series'>
DIM COL: y <class 'str'> <class 'pandas.core.series.Series'>
DIM COL: z <class 'str'> <class 'pandas.core.series.Series'>
NUM CHECK COL: carat <class 'str'> <class 'pandas.core.series.Series'>
NUM CHECK COL: depth <class 'str'> <class 'pandas.core.series.Series'>
NUM CHECK COL: table <class 'str'> <class 'pandas.core.series.Series'>
NUM CHECK COL: x <class 'str'> <class 'pandas.core.series.Series'>
NUM CHECK COL: y <class 'str'> <class 'pandas.core.series.Series'>
NUM CHECK COL: z <class 'str'> <class 'pandas.core.series.Series'>
NUM CHECK COL: price <class 'str'> <class 'pandas.core.series.Series'>


In [7]:
diagnostics = detect_missing_and_invalid_values(
    df = raw_df,
    numeric_cols = schema["numeric_cols"],
    categorical_cols = schema["categorical_cols"],
    target_col = schema["target_col"],
    dimension_cols = schema["dimension_columns"],
)

display(diagnostics["zero_xyz_summary"])
display(diagnostics["impossible_numeric_summary"])

,column,zero_count,zero_pct
0,x,8,0.0148
1,y,7,0.0130
2,z,20,0.0371


,column,non_positive_count,negative_count
0,carat,0,0
1,depth,0,0
2,table,0,0
3,x,8,0
4,y,7,0
5,z,20,0
6,price,0,0


In [8]:
dropped_columns = decide_columns_to_drop(raw_df, schema["expected_columns"])
print("Columns to drop:", dropped_columns)

Columns to drop: []


In [9]:
working_df = raw_df.drop(columns=dropped_columns, errors="ignore").copy()
working_df = mark_invalid_xyz_as_missing(working_df, schema["dimension_columns"])

cleaned_no_core_invalid_df, removed_rows_df = remove_impossible_core_rows(
    working_df,
    schema["strictly_positive_columns"],
)

print("Rows removed:", len(removed_rows_df))
display(removed_rows_df.head())

Rows removed: 0


,carat,cut,color,clarity,depth,table,price,x,y,z


In [10]:
cleaned_df, imputation_values = impute_missing_values(
    df=cleaned_no_core_invalid_df,
    numeric_cols=schema["numeric_cols"],
    categorical_cols=schema["categorical_cols"],
    numeric_strategy=schema["missing_strategy"]["numeric"],
    categorical_strategy=schema["missing_strategy"]["categorical"],
)

print("Imputation values:")
imputation_values

Imputation values:


{'carat': 0.7,
 'depth': 61.8,
 'table': 57.0,
 'x': 5.7,
 'y': 5.71,
 'z': 3.53,
 'cut': 'Ideal',
 'color': 'G',
 'clarity': 'SI1'}

In [11]:
missing_summary_after = build_missing_summary(cleaned_df)

print("Remaining total missing values:", int(cleaned_df.isna().sum().sum()))
display(missing_summary_after)
display(cleaned_df.head())

Remaining total missing values: 0


,column,missing_count,missing_pct
0,carat,0,0.0
1,clarity,0,0.0
2,color,0,0.0
3,cut,0,0.0
4,depth,0,0.0
5,price,0,0.0
6,table,0,0.0
7,x,0,0.0
8,y,0,0.0
9,z,0,0.0


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [12]:
numeric_df, categorical_df = split_numeric_and_categorical(cleaned_df, project_root)

print("Numeric shape:", numeric_df.shape)
print("Categorical shape:", categorical_df.shape)

display(numeric_df.head())
display(categorical_df.head())

Numeric shape: (53940, 6)
Categorical shape: (53940, 3)


,carat,depth,table,x,y,z
0,0.23,61.5,55.0,3.95,3.98,2.43
1,0.21,59.8,61.0,3.89,3.84,2.31
2,0.23,56.9,65.0,4.05,4.07,2.31
3,0.29,62.4,58.0,4.20,4.23,2.63
4,0.31,63.3,58.0,4.34,4.35,2.75


,cut,color,clarity
0,Ideal,E,SI2
1,Premium,E,SI1
2,Good,E,VS1
3,Premium,I,VS2
4,Good,J,SI2


In [13]:
preprocessing_summary = summarize_preprocessing_strategy(project_root)
preprocessing_summary

{'numeric_columns': ['carat', 'depth', 'table', 'x', 'y', 'z'],
 'categorical_columns': ['cut', 'color', 'clarity'],
 'dataset_level_cleaning_strategy': {'numeric_missing': 'median',
  'categorical_missing': 'most_frequent',
  'invalid_xyz': 'set 0/negative x,y,z to NaN, then impute',
  'impossible_core_rows': 'remove rows where carat <= 0 or price <= 0'},
 'regression_pipeline_strategy': {'numeric_imputation': 'median',
  'categorical_imputation': 'most_frequent',
  'categorical_encoding': 'onehot',
  'handle_unknown_categories': 'ignore',
  'scaling_for_linear_models': True},
 'clustering_pipeline_strategy': {'numeric_imputation': 'median',
  'categorical_imputation': 'most_frequent',
  'categorical_encoding': 'ordinal',
  'scaling': 'standard'}}

In [14]:
regression_preprocessor = build_regression_preprocessor(project_root)
clustering_preprocessor = build_clustering_preprocessor(project_root)

regression_preprocessor, clustering_preprocessor

(ColumnTransformer(transformers=[('num',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='median')),
                                                  ('scaler', StandardScaler())]),
                                  ['carat', 'depth', 'table', 'x', 'y', 'z']),
                                 ('cat',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='most_frequent')),
                                                  ('encoder',
                                                   OneHotEncoder(handle_unknown='ignore',
                                                                 sparse_output=False))]),
                                  ['cut', 'color', 'clarity'])],
                   verbose_feature_names_out=False),
 ColumnTransformer(transformers=[('num',
                                  Pipeline(steps=[('i

In [15]:
results = clean_diamonds_dataset(project_root)

print("Interim dataset saved to:", results["interim_output"])
print("Processed dataset saved to:", results["processed_output"])
print("Report saved to:", results["report_output"])
print("Missing-values figure saved to:", results["missing_fig_output"])
print("Invalid-xyz figure saved to:", results["invalid_xyz_fig_output"])

2026-03-21 01:17:24,696 | INFO | src.utils.logger | Loading raw dataset...
2026-03-21 01:17:24,955 | INFO | src.utils.logger | Building initial diagnostics...
2026-03-21 01:17:24,985 | INFO | src.utils.logger | Marking invalid x/y/z values are missing...
2026-03-21 01:17:24,993 | INFO | src.utils.logger | Removing impossible core rows...
2026-03-21 01:17:25,009 | INFO | src.utils.logger | Imputing remaining missing values...
2026-03-21 01:17:25,086 | INFO | src.utils.logger | Saving figures...
2026-03-21 01:17:26,022 | INFO | src.utils.logger | Saving markdown report...
2026-03-21 01:17:26,073 | INFO | src.utils.logger | Saving cleaned datasets...


Interim dataset saved to: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\data\interim\diamonds_cleaned.csv
Processed dataset saved to: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\data\processed\diamonds_processed.csv
Report saved to: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\reports\processing_report.md
Missing-values figure saved to: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\figures\preprocessing\missing_values_summary.png
Invalid-xyz figure saved to: F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\figures\preprocessing\invalid_xyz_analysis.png


In [16]:
processed_path = Path(results["processed_output"])
print(processed_path)
print("Exists:", processed_path.exists())

processed_df = pd.read_csv(processed_path)
print("Processed shape:", processed_df.shape)
display(processed_df.head())

print("Processed shape:", processed_df.shape)
display(processed_df.head())

F:\DATA SCIENCE\Projects\Diamond Dynamics Price Prediction and Market Segmentation\data\processed\diamonds_processed.csv
Exists: True
Processed shape: (53940, 10)


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


Processed shape: (53940, 10)


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75
